# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [3]:
# 1. Target distribution
fraud["fraud"].value_counts()

fraud
0.0    912597
1.0     87403
Name: count, dtype: int64

In [4]:
fraud["fraud"].value_counts(normalize=True) * 100

fraud
0.0    91.2597
1.0     8.7403
Name: proportion, dtype: float64

The target variable is imbalanced, with 91.26% legitimate transactions and only 8.74% fraudulent transactions.

In [5]:
# 2. Logistic Regression
features = fraud.drop(columns=["fraud"])
target = fraud["fraud"]

X_train, X_test, y_train, y_test = train_test_split(features,target,test_size=0.20,random_state=0)

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression()
log_reg.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [8]:
# 3. Evaluate the model
from sklearn.metrics import recall_score

y_pred = log_reg.predict(X_test_scaled)
print(f"Recall: {recall_score(y_test, y_pred):.2f}")

# Recall to minimize false negatives

Recall: 0.60


In [9]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98    182615
         1.0       0.89      0.60      0.72     17385

    accuracy                           0.96    200000
   macro avg       0.93      0.80      0.85    200000
weighted avg       0.96      0.96      0.96    200000



In [10]:
# 4. OverSample

train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
train["fraud"] = y_train.values

In [11]:
fraud_yes = train[train["fraud"] == 1]
fraud_no = train[train["fraud"] == 0]

In [12]:
len(fraud_yes), len(fraud_no)

(70018, 729982)

In [13]:
from sklearn.utils import resample
yes_oversampled = resample(fraud_yes,
                                    replace=True,
                                    n_samples = len(fraud_no),
                                    random_state=0)

In [14]:
train_over = pd.concat([yes_oversampled, fraud_no])
train_over

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
828609,-0.361720,0.209293,2.749213,0.366318,-0.734428,-0.334533,0.73251,1.0
112528,3.267297,-0.091417,0.353832,0.366318,-0.734428,-0.334533,0.73251,1.0
700883,0.050601,-0.063043,1.377882,0.366318,1.361603,-0.334533,0.73251,1.0
623474,-0.394272,0.095175,0.844943,-2.729870,-0.734428,-0.334533,-1.36517,1.0
84653,2.191924,-0.099482,-0.546546,0.366318,-0.734428,-0.334533,0.73251,1.0
...,...,...,...,...,...,...,...,...
963395,0.686916,-0.071705,-0.606513,0.366318,-0.734428,-0.334533,0.73251,0.0
117952,-0.359697,0.294316,0.106885,0.366318,-0.734428,-0.334533,0.73251,0.0
435829,0.002457,-0.023525,-0.477266,0.366318,-0.734428,2.989238,-1.36517,0.0
305711,-0.223923,-0.185220,-0.010973,0.366318,-0.734428,-0.334533,0.73251,0.0


In [15]:
train_over["fraud"].value_counts()

fraud
1.0    729982
0.0    729982
Name: count, dtype: int64

In [16]:
features_over = train_over.drop(columns=["fraud"])
target_over = train_over["fraud"]

In [17]:
log_reg_over = LogisticRegression()
log_reg_over.fit(features_over, target_over)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [18]:
y_pred_over = log_reg_over.predict(X_test_scaled)

C:\Users\carla\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [19]:
print(f"Recall: {recall_score(y_test, y_pred_over):.2f}")

Recall: 0.95


In [20]:
print(classification_report(y_test, y_pred_over))

              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182615
         1.0       0.57      0.95      0.71     17385

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



Oversampling improved the recall for fraudulent transactions from 0.60 to 0.95. However, accuracy decreased from 0.96 to 0.93 and precision for the fraud class decreased. The model detects many more fraud cases, but it also produces more false positives.

In [21]:
# 5. UnderSample
fraud_no_undersampled = resample(fraud_no,replace=False,n_samples=len(fraud_yes),random_state=0)
train_under = pd.concat([fraud_no_undersampled, fraud_yes])

In [22]:
train_under["fraud"].value_counts()

fraud
0.0    70018
1.0    70018
Name: count, dtype: int64

In [23]:
features_under = train_under.drop(columns=["fraud"])
target_under = train_under["fraud"]

In [24]:
log_reg_under = LogisticRegression()
log_reg_under.fit(features_under, target_under)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [25]:
y_pred_under = log_reg_under.predict(X_test_scaled)

C:\Users\carla\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [26]:
print(f"Recall: {recall_score(y_test, y_pred_under):.2f}")

Recall: 0.95


In [27]:
print(classification_report(y_test, y_pred_under))

              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182615
         1.0       0.57      0.95      0.71     17385

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



Undersampling gives the same results as oversampling, with a recall of 0.95 for the fraud class and an accuracy of 0.93. Therefore, in this case, undersampling does not improve the performance compared with oversampling.

In [28]:
# 6. SMOTE
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=0)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train)

In [29]:
y_train_smote.value_counts()

fraud
0.0    729982
1.0    729982
Name: count, dtype: int64

In [30]:
log_reg_smote = LogisticRegression()
log_reg_smote.fit(X_train_smote, y_train_smote)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [31]:
y_pred_smote = log_reg_smote.predict(X_test_scaled)

In [32]:
print(f"Recall: {recall_score(y_test, y_pred_smote):.2f}")

Recall: 0.95


In [33]:
print(classification_report(y_test, y_pred_smote))

              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182615
         1.0       0.57      0.95      0.71     17385

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



Oversampling, undersampling and SMOTE improved the recall of fraudulent transactions from 0.60 to 0.95. However, accuracy decreased from 0.96 to 0.93 and precision for the fraud class decreased from 0.89 to 0.57. In this case, the balancing techniques helped the model detect more fraud, but also increased the number of false positives.